In [1]:
import os
import pandas as pd
import psycopg2
from datetime import datetime, timedelta

In [ ]:
if (os.environ.get('TINKOFF_API_TOKEN', '') == ''):
    print('Environment variable TINKOFF_API_TOKEN not setted!!')
else:
    TOKEN = os.environ["TINKOFF_API_TOKEN"]

if (os.environ.get('DB_USER', '') == ''):
    print('Environment variable DB_USER not setted!!')
else:
    DB_USER= os.environ["DB_USER"]

if (os.environ.get('DB_PASSWORD', '') == ''):
    print('Environment variable DB_PASSWORD not setted!!')
else:
    DB_PASSWORD = os.environ["DB_PASSWORD"]

if (os.environ.get('DB_NAME', '') == ''):
    print('Environment variable DB_NAME not setted!!')
else:
    DB_NAME = os.environ["DB_NAME"]

if (os.environ.get('DB_HOST', '') == ''):
    print('Environment variable DB_HOST not setted!!')
else:
    DB_HOST = os.environ["DB_HOST"]

if (os.environ.get('DB_PORT', '') == ''):
    print('Environment variable DB_PORT not setted!!')
else:
    DB_PORT = os.environ["DB_PORT"]



In [3]:
def save_to_db(df, table_name):
    try:
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            host=DB_HOST,
            port=DB_PORT
        )
   
        cursor = conn.cursor()
        cursor.execute(f'''
            CREATE TABLE IF NOT EXISTS {table_name} (
                time TIMESTAMP PRIMARY KEY,
                open FLOAT,
                high FLOAT,
                low FLOAT,
                close FLOAT,
                volume BIGINT,
                Close_Lag_1 FLOAT NULL,
                Close_Lag_2 FLOAT NULL,
                Close_Lag_3 FLOAT NULL,
                Rolling_Mean_7 FLOAT NULL,
                Rolling_Std_7 FLOAT NULL,
                Rolling_Max_7 FLOAT NULL,
                Rolling_Min_7 FLOAT NULL,
                SMA_14 FLOAT NULL
            )
        ''')

        for index, row in df.iterrows():
                cursor.execute(f'''
                    INSERT INTO {table_name} (time, open, high, low, close, volume, Close_Lag_1, Close_Lag_2, Close_Lag_3, Rolling_Mean_7, Rolling_Std_7, Rolling_Max_7, Rolling_Min_7, SMA_14)
                    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (time) DO NOTHING
                ''', (
                    row['time'].strftime('%Y-%m-%d %H:%M:%S'), 
                    row['open'],
                    row['high'],
                    row['low'],
                    row['close'],
                    row['volume'],
                    row['Close_Lag_1'],
                    row['Close_Lag_2'],
                    row['Close_Lag_3'],
                    row['Rolling_Mean_7'],
                    row['Rolling_Std_7'],
                    row['Rolling_Max_7'],
                    row['Rolling_Min_7'],
                    row['SMA_14']
                ))
        conn.commit()
    except psycopg2.OperationalError as e:
        print(f"Error save_data, OperationalError: {e}")
        raise ConnectionError(f"Error connect to DB: {str(e)}")
    except Exception as e:
        print(f"Error save_data: {e}")
        return None

    finally:
        if conn:
            cursor.close()
            conn.close()

In [4]:
def get_data(depth_days: int, table_name: str):
    try:
        date = datetime.now()
        date = date - timedelta(days=depth_days)
        
        conn = psycopg2.connect(
            dbname=DB_NAME,
            user=DB_USER,
            password=DB_PASSWORD,
            host=DB_HOST,
            port=DB_PORT
        )
        cursor = conn.cursor()

        query = f"""
            SELECT time, open, high, low, close, volume, 
                   Close_Lag_1, Close_Lag_2, Close_Lag_3, 
                   Rolling_Mean_7, Rolling_Std_7, Rolling_Max_7, Rolling_Min_7, SMA_14
            FROM {table_name}
            WHERE time > '{date}'
            ORDER BY time 

        """
        cursor.execute(query)
        result = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]

        df = pd.DataFrame(result, columns=columns)

        return df
    except psycopg2.OperationalError as e:
        raise ConnectionError(f"Error connect to DB: {str(e)}")
    except Exception as e:
        print(f"Error get_data: {e}")
        return None

    finally:
        if conn:
            cursor.close()
            conn.close()